# 01 — Exploring and qualifying the sources

This notebook checks **concretely**, source by source, that the announced data really is
there: a text, an image, and where applicable a label. The full reasoning — why these
sources, why no scraping — is in `docs/source_exploration.md`.

In [1]:
import sys
from pathlib import Path

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RACINE / "src"))

from dotenv import load_dotenv

load_dotenv(RACINE / ".env")

from multimodal_etl.logging_setup import setup_logging

setup_logging()

## The four sources and their access methods

A pipeline source is a module under `multimodal_etl.sources` exposing a `fetch_*`
function, plus one line in the connector table. Each goes through a different channel, and
that is deliberate: it avoids depending on a single way in.

In [2]:
from multimodal_etl.extract import _CONNECTORS

for nom, fonction in _CONNECTORS.items():
    print(f"{nom:18s} -> {fonction.__module__}")

rss                -> multimodal_etl.sources.rss
newsdata           -> multimodal_etl.sources.newsdata
fakenewsnet        -> multimodal_etl.sources.fakenewsnet
kaggle_fakeddit    -> multimodal_etl.sources.kaggle_fakeddit


## 1. RSS feeds — the multimodal backbone

RSS feeds are published by the editors *to be redistributed*: free, no key, updated
continuously. The difficulty lies elsewhere — **the image is never in the same place**
from one publisher to the next. The connector looks for it in `media:content`,
`media:thumbnail`, the attachments, then in the summary's HTML.

In [3]:
from multimodal_etl.config import ExtractionConfig
from multimodal_etl.sources.rss import fetch_rss_feed

config = ExtractionConfig()
flux = dict(config.rss_feeds)
publications = fetch_rss_feed("bbc_news", flux["bbc_news"], config)

avec_image = [p for p in publications if p["image_url"]]
print(f"{len(publications)} publications lues, dont {len(avec_image)} avec une image")

exemple = avec_image[0]
for cle in ("title", "url", "image_url", "access_method"):
    print(f"{cle:14s}: {str(exemple[cle])[:88]}")

2026-08-20 12:01:13 | INFO    | multimodal_etl.sources.rss | RSS : lecture du flux 'bbc_news' (https://feeds.bbci.co.uk/news/world/rss.xml)


2026-08-20 12:01:14 | INFO    | multimodal_etl.sources.rss | RSS : 27 publications recuperees depuis 'bbc_news'


27 publications lues, dont 27 avec une image
title         : At least 13 killed in Kyiv as Ukraine grapples with air defence shortages
url           : https://www.bbc.co.uk/news/articles/c98vzmden5yo?at_medium=RSS&at_campaign=rss
image_url     : https://ichef.bbci.co.uk/ace/standard/240/cpsprodpb/630f/live/b3d19af0-9c6e-11f1-8efc-bf
access_method : flux_rss


## 2. The NewsData.io API — news already normalised

The API returns JSON with an explicit `image_url` field. In exchange it imposes a daily
quota: the connector reads a single page, and **turns itself off cleanly** when no key is
supplied rather than failing the pipeline.

In [4]:
from multimodal_etl.sources import newsdata

print("Clé API disponible :", newsdata.is_enabled())
articles = newsdata.fetch_newsdata(config)
print(f"{len(articles)} articles récupérés")
if articles:
    print("Titre :", articles[0]["title"][:88])
    print("Image :", articles[0]["image_url"][:88])

Clé API disponible : True
2026-08-20 12:01:14 | INFO    | multimodal_etl.sources.newsdata | NewsData.io : appel de l'API (https://newsdata.io/api/1/news)


2026-08-20 12:01:14 | INFO    | multimodal_etl.sources.newsdata | NewsData.io : 10 articles recuperes


10 articles récupérés
Titre : School aide charged in assault of disabled student
Image : https://bloximages.chicago2.vip.townnews.com/jeffcotranscript.com/content/tncms/assets/v


## 3. FakeNewsNet — a labelled dataset, but with no image

The CSVs published on GitHub hold `id, news_url, title, tweet_ids`. There is **no image**
at all: something the dataset's own description does not say outright.

Two practical consequences the connector handles:
1. the `tweet_ids` column exceeds the field size the `csv` module accepts by default — the
   limit has to be raised, or the read fails;
2. the image has to be found elsewhere: in the `og:image` tag the publisher itself puts on
   the article's page.

In [5]:
from multimodal_etl.sources import fakenewsnet

path_for = fakenewsnet.download_csv("politifact_fake.csv", config)
lignes = fakenewsnet.read_csv(path_for)
print("Colonnes réelles du fichier :", list(lignes[0].keys()))
print(f"{len(lignes)} lignes labellisées disponibles")
print("Exemple de titre :", lignes[0]["title"][:88])

2026-08-20 12:01:14 | INFO    | multimodal_etl.sources.fakenewsnet | FakeNewsNet : 'politifact_fake.csv' déjà en cache


Colonnes réelles du fichier : ['id', 'news_url', 'title', 'tweet_ids']
432 lignes labellisées disponibles
Exemple de titre : BREAKING: First NFL Team Declares Bankruptcy Over Kneeling Thugs


### What the Open Graph enrichment actually yields

We measure what we really recover: the PolitiFact URLs date from 2016-2018 and many no
longer answer. That is a constraint to know about, not a flaw to hide — publications
without an image are dropped at the transform step.

In [6]:
from multimodal_etl.sources import opengraph

echantillon = [fakenewsnet._build_record(ligne, "politifact", "fake") for ligne in lignes[:10]]
compteurs = opengraph.enrich_publications(echantillon, config)
print(compteurs)

2026-08-20 12:01:38 | INFO    | multimodal_etl.sources.opengraph | Open Graph : 2 images retrouvées sur 10 articles consultés


{'attempted': 10, 'found': 2}


## 4. Fakeddit — the multimodal dataset from Kaggle

Fakeddit natively pairs a title and an image, with three levels of labels. The file is
downloaded once by hand from Kaggle and dropped into `data/raw/kaggle/`; in its absence,
the connector reads a versioned demonstration sample of identical structure.

In [7]:
import pandas as pd

from multimodal_etl.sources import kaggle_fakeddit

publications_kaggle = kaggle_fakeddit.fetch_fakeddit(config)
df_kaggle = pd.DataFrame(publications_kaggle)
print(f"{len(df_kaggle)} publications chargées")
df_kaggle["label"].value_counts()

2026-08-20 12:01:40 | INFO    | multimodal_etl.sources.kaggle_fakeddit | Fakeddit : jeu Kaggle absent, utilisation de fakeddit_sample.tsv


2026-08-20 12:01:40 | INFO    | multimodal_etl.sources.kaggle_fakeddit | Fakeddit : 24 publications chargées depuis fakeddit_sample.tsv (échantillon de démonstration)


24 publications chargées


label
fake    12
real    12
Name: count, dtype: int64

## Summary

The four sources complement each other: the RSS feeds and the API bring **volume and
freshness**, FakeNewsNet and Fakeddit bring **labels**. None goes through scraping: they
are all channels the data producer intended for this use.